## 7.3 Simple RNN实现 - 完整案例

#### 1、案例目标与数据集选择

##### 1.1 我们这次要完成什么
我们用一个完整的文本分类案例，把前面学过的内容全部串起来：

- 获取经典数据集
- 做文本预处理
- 构建词表
- 把文本变成 RNN 可接收的数字序列
- 搭建 Simple RNN 模型
- 完成训练、验证、测试
- 最后做单条文本预测

这个案例最适合放在我们当前阶段，因为它刚好能把前面学过的：

- 自然语言预处理
- token / vocab / padding
- `nn.RNN`
- 前向传播与隐藏状态
- 交叉熵损失函数
- 训练循环与评估流程

全部连起来。

##### 1.2 数据集选择
这里使用 IMDb Large Movie Review Dataset 来做情感分类。

选择它的原因很经典，也很适合入门：

- 任务简单明确：二分类（positive / negative）
- 文本是真实英文影评，很适合做 NLP 入门
- 数据量足够大，能体现 RNN 的训练流程
- 官方数据集包含 25,000 条训练样本 和 25,000 条测试样本，另外还有未标注数据；
- Hugging Face 也提供了方便加载的 imdb 数据集接口。

##### 1.3 本案例最终要做的任务
输入一段电影评论文本，例如：

- `“This movie was amazing and touching.”`
- `“The plot was boring and the acting was terrible.”`

模型输出：

- 正面情感 positive
- 负面情感 negative

这本质上就是一个 many-to-one 的序列分类任务：输入是一整段单词序列，输出是一个类别标签。

#### 2、完整流程总览

##### 2.1 整体流程图
原始文本  
$\rightarrow$ 分词  
$\rightarrow$ 建立词表  
$\rightarrow$ token 转 id  
$\rightarrow$ 截断 / 补齐长度  
$\rightarrow$ 组成 batch  
$\rightarrow$ 输入 RNN  
$\rightarrow$ 取最后时刻输出  
$\rightarrow$ 全连接分类  
$\rightarrow$ 计算损失  
$\rightarrow$ 反向传播更新参数

##### 2.2 我们这次采用的技术方案
为了让案例既完整又不复杂，这里采用下面这一套：

- 数据集加载：`datasets` 库中的 IMDb
- 分词方式：最基础的 `lower().split()`
- 词表：自己手动构建
- 长度处理：固定长度 `max_len`
- 模型：Embedding + Simple RNN + Linear
- 损失函数：`CrossEntropyLoss`
- 优化器：`Adam`
- 学习率迭代：`ReduceLROnPlateau`


#### 3、环境准备与安装

##### 3.1 需要的库
```python
pip install torch datasets scikit-learn
```

这里使用 Hugging Face `datasets` 来加载 IMDb 数据集。  
官方文档说明，可以通过 `load_dataset()` 一行代码加载 Hub 上的数据集。

##### 3.2 导入依赖

In [1]:
import re
import random
from collections import Counter # Counter 用于统计词频的出现次数

import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.model_selection import train_test_split

#### 4、数据集获取与查看

##### 4.1 加载 IMDb 数据集


你会得到类似结构：

- `train`
- `test`

其中每条样本主要包含：

- `text`
- `label`

IMDb 在 Hugging Face 上的常见结构是：

- `train`：25,000
- `test`：25,000

标签字段为 `text` 和 `label`。

In [2]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 205553.36 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


##### 4.2 查看一条样本

可能输出类似：

```python
{
    'text': 'I rented I AM CURIOUS-YELLOW from my video store because ...',
    'label': 0
}
```

其中：

- `text`：影评文本
- `label`：类别标签
- `0` 通常表示 negative
- `1` 通常表示 positive

In [3]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

##### 4.3 为什么还要再拆分验证集
IMDb 默认一般给我们的是：

- 训练集 `train`
- 测试集 `test`

但是在真正训练模型时，我们通常还需要再从训练集里划出一部分做验证集 `validation`，用于：

- 调参
- 观察是否过拟合
- 保存最佳模型

常见做法是：

`train` 原始 25,000 条中，再切一部分出来作为 `val`

例如：

- `train`: 20,000
- `val`: 5,000
- `test`: 25,000

#### 5、划分训练集 / 验证集 / 测试集

##### 5.1 先取出原始数据

In [4]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

##### 5.2 从训练集再切出验证集

In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels # 分层抽样，保持训练集和验证集中各类别的比例与原始数据集相同
)

##### 5.3 看一下数据规模

In [6]:
print("Train size: ", len(train_texts))
print("Validation size: ", len(val_texts))
print("Test size: ", len(test_texts))

Train size:  20000
Validation size:  5000
Test size:  25000


#### 6、文本预处理方法定义

##### 6.1 先明确：我们这里不用复杂预处理
这次我们只使用最基础、最适合教学的处理方式：

- 全部转小写
- 去掉 HTML 标签
- 去掉多余符号
- 按空格分词

这样做的目的不是最优，而是为了让你看清楚：

原始文本是怎么一步一步变成数字张量的。

##### 6.2 文本清洗函数

In [7]:
def clean_text(text):
    text = text.lower() 
    text = re.sub(r"<br\s*/?>", " ", text) # 替换HTML标签为一个空格
    text = re.sub(r"[^a-zA-Z\s]", "", text) # 保留字母、数字和空格，单引号
    text = re.sub(r"\s+", " ", text).strip() # 替换多个空格为一个空格，并去除首尾空格
    return text

##### 6.3 分词函数

例如：

```python
text = "This movie is REALLY great!!! <br /> I love it."
tokens = tokenize(text)
print(tokens)
```

输出可能是：

```python
['this', 'movie', 'is', 'really', 'great', 'i', 'love', 'it']
```

In [8]:
def tokenize(text):
    return clean_text(text).split() # 先调用clean_text进行清洗，然后使用split方法将文本分割成单词列表

##### 6.4 为什么 `split()` 的结果是 `list[list[token]]`
这是前面小节提到过的一个关键理解点。

一条句子分词后，是：`list[token]`  
一个数据集里有很多句子，所以整体会变成：`list[list[token]]`

例如：

```python
sentences = [
    "I love this movie",
    "It is very boring"
]
```

处理后会变成：

```python
[
    ['i', 'love', 'this', 'movie'],
    ['it', 'is', 'very', 'boring']
]
```

所以：

- 单句：token 列表
- 整个数据集：句子列表，每个句子里面再是 token 列表

#### 7、构建词表 Vocabulary

##### 7.1 先统计训练集词频
注意：词表只能用训练集构建，不能偷看验证集和测试集。

In [9]:
def build_vocab(texts, min_freq=5): # min_freq参数用于设置词频的最低阈值，只有出现次数大于或等于min_freq的单词才会被保留在词汇表中
    counter = Counter()
    for text in texts:
        tokens = tokenize(text) # 调用清洗 + 分词之后得到的单词列表
        counter.update(tokens) # 更新Counter对象，统计每个单词的出现次数

    # 创建空的词汇表，并添加特殊标记
    vocab = {
        "<PAD>": 0, # 用于填充序列，使其具有相同的长度
        "<UNK>": 1  # 用于表示词汇表中未出现的单词
    }

    # 遍历Counter对象中的单词和对应的频率
    for word, freq in counter.items():
        if freq >= min_freq: # 只有当单词的频率大于或等于min_freq时，才将其添加到词汇表中
            vocab[word] = len(vocab) # 将单词添加到词汇表中，并为其分配一个唯一的索引，这里使用词汇表的长度也就是当前词汇的顺序作为id
    return vocab

##### 7.2 构建词表

In [10]:
vocab = build_vocab(train_texts, min_freq=2)
print("Vocabulary size: ", len(vocab))

Vocabulary size:  47524


##### 7.3 为什么要有 `<PAD>` 和 `<UNK>`
- `<PAD>`：用于补齐长度
- `<UNK>`：表示未知词，测试时遇到没见过的词，就映射成它

这是文本任务中非常基础的两个特殊 token。

#### 8、把文本转换为数字序列

##### 8.1 文本转 id

例如：

```python
sample = "this movie is wonderful"
ids = text_to_ids(sample, vocab)
print(ids)
```

In [ ]:
# 方法流程：
# 1. 首先再次调用tokenize函数对输入文本进行清洗和分词，得到一个单词列表。
# 2.1 然后使用列表推导式遍历这个单词列表，对于每个单词，使用vocab.get(token, vocab["<UNK>"])来获取其对应的id。
# 2.2 由于测试集中的文本可能不存在于训练集得到的vocab中，如果单词在词汇表中存在，就返回对应的id；如果单词不在词汇表中，就返回vocab["<UNK>"]的id，即表示未知单词。
# 3. 最后返回一个由单词id组成的列表，这个列表可以用作模型的输入。
def text_to_ids(text, vocab):
    tokens = tokenize(text) # 调用清洗 + 分词之后得到的单词列表
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens] # 将每个单词转换为对应的id，如果单词不在词汇表中，则使用<UNK>的id

##### 8.2 为什么还要固定长度
因为 `DataLoader` 组 batch 时，通常要求同一个 batch 里的样本 shape 一致。

但文本长度天然不一样：

- 有的评论 20 个词
- 有的评论 200 个词
- 有的评论 1000 个词

##### 8.3 截断与补齐函数

In [ ]:
def pad_or_truncate(token_ids, max_len, pad_id=0): # 这个函数的作用是对输入的token_ids列表进行填充或截断，使其长度达到max_len。
    if len(token_ids) > max_len:
        return token_ids[:max_len] # 如果token_ids的长度超过max_len，则截断为max_len长度的子列表
    else:
        return token_ids + [pad_id] * (max_len - len(token_ids)) # 如果token_ids的长度不足max_len，则在其后面添加pad_id（默认为0）直到达到max_len长度

#### 9、使用前面的所有方法组装自定义 Dataset 与 DataLoader

##### 9.1 自定义 Dataset

这里标签使用 `long`，因为 PyTorch 的 `CrossEntropyLoss` 在使用类别索引时，target 需要是整型类别索引。  

官方文档说明，输入是 logits，target 使用 class indices 时需要是 `long` 类型，类别索引范围是 `[0, C)`。

In [ ]:
# 为什么要自己组装Dataset？
# 1. 数据预处理：在文本分类任务中，通常需要对文本数据进行清洗、分词、构建词汇表、将文本转换为数值表示（如单词id列表）等预处理步骤。通过自定义Dataset类，可以将这些预处理步骤封装在一起，使得数据加载和预处理更加高效和便捷。
# 2. 灵活性：自定义Dataset类可以根据具体的任务需求进行定制化设计，例如可以添加额外的特征、处理不同类型的数据等。这种灵活性使得模型训练过程更加适应特定的任务和数据。
# 3. 与DataLoader配合使用：PyTorch的DataLoader可以与自定义的Dataset类无缝配合，提供批量加载数据、打乱数据顺序、并行加载等功能。这使得训练过程更加高效，尤其是在处理大型数据集时。
class IMDBDataset(Dataset):
    # init方法用于初始化数据集对象，接受文本数据、标签、词汇表和最大长度等参数，并将它们存储为类的属性。
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
    # len方法返回数据集的大小，即文本数据的数量。
    def __len__(self):
        return len(self.texts)
    # getitem方法用于获取数据集中的一个样本，接受一个索引参数idx，返回对应文本的单词id列表和标签的PyTorch张量。
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        token_ids = text_to_ids(text, self.vocab) # 将文本转换为单词id列表
        token_ids = pad_or_truncate(token_ids, self.max_len) # 对单词id列表进行填充或截断，使其长度达到max_len

        return (
            torch.tensor(token_ids, dtype=torch.long), # 将处理后的单词id列表转换为PyTorch张量，数据类型为long
            torch.tensor(label, dtype=torch.long) # 将标签转换为PyTorch张量，数据类型为long
        )

##### 9.2 创建数据集对象

In [16]:
max_len = 200
train_dataset = IMDBDataset(train_texts, train_labels, vocab, max_len)
val_dataset = IMDBDataset(val_texts, val_labels, vocab, max_len)
test_dataset = IMDBDataset(test_texts, test_labels, vocab, max_len)

##### 9.3 创建 DataLoader

In [17]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

##### 9.4 检查一个 batch 的 shape

In [18]:
X_batch, y_batch = next(iter(train_loader))
print("Batch X shape: ", X_batch.shape)
print("Batch y shape: ", y_batch.shape)

Batch X shape:  torch.Size([64, 200])
Batch y shape:  torch.Size([64])


#### 10、Simple RNN 模型构建

##### 10.1 模型结构设计
我们这里使用：

- Embedding
- `nn.RNN`
- Linear

流程是：

词 id  
$\rightarrow$ Embedding 变成词向量  
$\rightarrow$ RNN 逐时间步处理  
$\rightarrow$ 取最后一个时间步的输出  
$\rightarrow$ 全连接层输出类别分数

##### 10.2 回顾 PyTorch 的 `nn.RNN`
PyTorch 官方文档说明：

如果 `batch_first=True`

输入 shape 是 `(batch, seq_len, input_size)`  
输出 `output shape` 是 `(batch, seq_len, hidden_size)`  
最终隐藏状态 `h_n shape` 是 `(num_layers * num_directions, batch, hidden_size)`。

所以对于我们的文本分类任务：

- `input_size = embedding_dim`
- `hidden_size = rnn_hidden_size`

最后用最后时刻的信息做分类。

##### 10.3 模型搭建

In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 嵌入层，将单词id转换为对应的嵌入向量
        self.rnn = nn.RNN(
            input_size=embedding_dim, # 输入维度等于嵌入维度
            hidden_size=hidden_dim, # 隐藏状态的维度
            batch_first=True # 输入和输出的张量格式为(batch_size, seq_len, feature_dim)
        )
        self.fc = nn.Linear(hidden_dim, num_classes) # 全连接层，将RNN的输出映射到类别数

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        x = self.embedding(x) # 将输入的单词id转换为嵌入向量，shape: [batch_size, seq_len, embedding_dim]
        output, h_n = self.rnn(x) # RNN层，output shape: [batch_size, seq_len, hidden_dim], h_n shape: [1, batch_size, hidden_dim]

        # 取最后一个时间步的输出作为文本的表示
        last_output = output[:, -1, :] # shape: [batch_size, hidden_dim]
        logits = self.fc(last_output) # 全连接层，shape: [batch_size, num_classes]
        return logits # 直接返回logits，crossEntropyLoss会在内部计算softmax和损失值

##### 10.4 为什么这里不手动加 Softmax
因为 `CrossEntropyLoss` 接收的是 logits，不是已经 softmax 过的概率。  
官方文档明确写的是 “cross entropy loss between input logits and target”；  
另外 `NLLLoss` 文档也说明，如果不想自己加 `LogSoftmax`，可以直接用 `CrossEntropyLoss`。

所以这里正确写法是：

- 模型最后直接输出 logits
- 不手动加 `Softmax`

#### 11、定义 device、损失函数、优化器、学习率迭代器

##### 11.1 设备选择

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


##### 11.2 实例化模型

In [21]:
vocab_size = len(vocab) # 词汇表的大小，即不同单词的数量，在之前已经构建了词汇表并计算了其大小
embedding_dim = 128 # 嵌入维度，表示每个单词将被映射到一个128维的向量空间中
hidden_dim = 128 # RNN隐藏状态的维度，表示RNN层中隐藏状态的大小
num_classes = 2 # 类别数，IMDB数据集是一个二分类任务，分别表示正面和负面评论

model = SimpleRNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
).to(device)

##### 11.3 定义损失函数与优化器还有学习率迭代器

In [22]:
criterion = nn.CrossEntropyLoss() # 交叉熵损失函数，适用于多分类任务
optimizer = torch.optim.Adam(
    model.parameters(), # 传入模型的参数，优化器将更新这些参数以最小化损失函数
    lr=1e-3 # 学习率，控制优化器更新参数的步长
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode="min",
    factor=0.1, # 当监测的指标停止改善时，学习率将乘以这个因子
    patience=5, # 当监测的指标停止改善时，等待多少个epoch后才调整学习率
    verbose=True # 是否打印学习率调整的日志信息
)

/home/zhang/miniconda3/envs/da/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


#### 12、训练与验证函数

##### 12.1 计算准确率函数

In [23]:
def calculate_accuracy(logits, labels): # 根据预测值 logits 和真实标签 labels 计算accuracy
    preds = torch.argmax(logits, dim=1) # 由logits通过softmax计算每个类别的概率， 然后使用argmax获取概率最高的类别索引作为预测结果
    correct = (preds == labels).sum().item() # 计算预测正确的数量，比较预测结果和真实标签是否相等，并求和得到正确的数量
    total = labels.size(0) # 获取标签的总数量，即批次大小
    return correct / total # 返回准确率，即正确的数量除以总数量

##### 12.2 训练函数

In [24]:
def train_one_epoch(model, criterion, optimizer, train_loader, device):
    model.train() # 将模型设置为训练模式，启用dropout和batch normalization等训练特定的行为
    total_loss = 0.0 # 初始化总损失
    total_acc = 0.0 # 初始化总准确率
    total_samples = 0 # 初始化总样本数量
    for X_batch, y_batch in train_loader: 
        X_batch, y_batch = X_batch.to(device), y_batch.to(device) # 将输入数据和标签移动到指定的设备（CPU或GPU）

        optimizer.zero_grad() # 清除之前的梯度信息，准备进行新的反向传播
        logits = model(X_batch) # 前向传播，得到模型的输出logits
        loss = criterion(logits, y_batch) # 计算损失值，比较模型的输出logits和真实标签y_batch之间的差异
        
        loss.backward() # 反向传播，计算损失函数相对于模型参数的梯度
        optimizer.step() # 更新模型参数，使用计算得到的梯度进行优化

        total_loss += loss.item() * X_batch.size(0) # 将当前批次的损失值乘以批次大小（样本数量）累加到total_loss中

        # 调用calculate_accuracy函数计算当前批次的准确率，并累加到total_acc中
        acc = calculate_accuracy(logits, y_batch) # 计算当前批次的准确率
        total_acc += acc * y_batch.size(0) # 将当前批次的准确率乘以批次大小（标签的数量）累加到total_acc中
        
        total_samples += X_batch.size(0) # 累加当前批次的样本数量到total_samples中
    
    avg_loss = total_loss / total_samples # 计算平均损失值，total_loss除以总样本数量
    avg_acc = total_acc / total_samples # 计算平均准确率，total_acc除
    
    return avg_loss, avg_acc # 返回平均损失值和平均准确率

##### 12.3 验证函数

In [25]:
@torch.no_grad() # 在评估模型时，不需要计算梯度，因此使用torch.no_grad()上下文管理器来禁用梯度计算，以节省内存和提高计算效率
def evaluate(model, criterion, val_loader, device):
    model.eval() # 将模型设置为评估模式，禁用dropout和batch normalization等训练特定的行为
    total_loss = 0.0 # 初始化总损失
    total_acc = 0.0 # 初始化总准确率
    total_samples = 0 # 初始化总样本数量
    for X_batch, y_batch in val_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device) # 将输入数据和标签移动到指定的设备（CPU或GPU）

        logits = model(X_batch) # 前向传播，得到模型的输出logits
        loss = criterion(logits, y_batch) # 计算损失值，比较模型的输出logits和真实标签y_batch之间的差异

        total_loss += loss.item() * X_batch.size(0) # 将当前批次的损失值乘以批次大小（样本数量）累加到total_loss中

        acc = calculate_accuracy(logits, y_batch) # 计算当前批次的准确率
        total_acc += acc * y_batch.size(0) # 将当前批次的准确率乘以批次大小（标签的数量）累加到total_acc中
        
        total_samples += X_batch.size(0) # 累加当前批次的样本数量到total_samples中
    
    avg_loss = total_loss / total_samples # 计算平均损失值，total_loss除以总样本数量
    avg_acc = total_acc / total_samples # 计算平均准确率，total_acc除以总样本数量
    
    return avg_loss, avg_acc # 返回平均损失值和平均准确率

#### 13、主训练循环

##### 13.1 训练若干轮
```python
num_epochs = 5
best_val_acc = 0.0
best_model_path = "best_simple_rnn_imdb.pt"

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 50)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
```

In [26]:
num_epochs = 30 
best_val_acc = 0.0 # 初始化最佳验证准确率
best_model_path = "best_simple_rnn_model.pth" # 定义保存最佳模型的路径

# 开始训练
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, criterion, optimizer, train_loader, device) # 训练一个epoch，返回训练损失和训练准确率
    val_loss, val_acc = evaluate(model, criterion, val_loader, device) # 在验证集上评估模型，返回验证损失和验证准确率

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    scheduler.step(val_loss) # 更新学习率调度器，根据验证损失调整学习率

    # 如果当前验证准确率比之前的最佳验证准确率更高，则保存当前模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc # 更新最佳验证准确率
        torch.save(model.state_dict(), best_model_path) # 保存当前模型的状态字典到指定路径

Epoch 1/30 - Train Loss: 0.7031, Train Acc: 0.4984, Val Loss: 0.7024, Val Acc: 0.5026
Epoch 2/30 - Train Loss: 0.6905, Train Acc: 0.5280, Val Loss: 0.6964, Val Acc: 0.5078
Epoch 3/30 - Train Loss: 0.6781, Train Acc: 0.5466, Val Loss: 0.7005, Val Acc: 0.5148
Epoch 4/30 - Train Loss: 0.6574, Train Acc: 0.5750, Val Loss: 0.7133, Val Acc: 0.5228
Epoch 5/30 - Train Loss: 0.6130, Train Acc: 0.6110, Val Loss: 0.7380, Val Acc: 0.5358
Epoch 6/30 - Train Loss: 0.5690, Train Acc: 0.6322, Val Loss: 0.8132, Val Acc: 0.5334
Epoch 7/30 - Train Loss: 0.5537, Train Acc: 0.6494, Val Loss: 0.8148, Val Acc: 0.5020
Epoch 8/30 - Train Loss: 0.5464, Train Acc: 0.6497, Val Loss: 0.9112, Val Acc: 0.4986
Epoch 9/30 - Train Loss: 0.4711, Train Acc: 0.6869, Val Loss: 0.9410, Val Acc: 0.5050
Epoch 10/30 - Train Loss: 0.4615, Train Acc: 0.6921, Val Loss: 0.9694, Val Acc: 0.4990
Epoch 11/30 - Train Loss: 0.4539, Train Acc: 0.6997, Val Loss: 0.9928, Val Acc: 0.5026
Epoch 12/30 - Train Loss: 0.4479, Train Acc: 0.6956,

##### 13.2 为什么要保存最佳模型
因为训练越往后，不一定验证集效果越好。  
有时会出现：

- train acc 继续升高
- val acc 开始下降

这就是过拟合信号。

所以我们通常保存：

验证集表现最好的那一轮模型

而不是最后一轮。


#### 14、测试集评估

##### 14.1 加载最佳模型

In [27]:
model.load_state_dict(torch.load(best_model_path)) # 加载之前保存的最佳模型的状态字典

/tmp/ipykernel_29398/139544293.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path)) # 加载之前保存的最佳模型的状态字典


<All keys matched successfully>

##### 14.2 在测试集上评估

In [28]:
test_loss, test_acc = evaluate(model, criterion, test_loader, device) # 在测试集上评估模型，返回测试损失和测试准确率
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Test Loss: 1.0438, Test Acc: 0.5844


##### 14.3 如何理解这个结果
如果是最基础的：

- 简单分词
- 手写词表
- Simple RNN

那么能得到一个还不错的二分类结果，就已经说明流程是成功的。

这里我们的重点不是刷分，而是：

✅ 完整理解从文本到模型输出的全过程

#### 15、用模型进行测试

##### 15.1 写预测函数

In [32]:
@torch.no_grad() # 在评估模型时，不需要计算梯度，因此使用torch.no_grad()上下文管理器来禁用梯度计算，以节省内存和提高计算效率
def predict_sentiment(text, model, max_len, device):
    model.eval()

    token_ids = text_to_ids(text, vocab) # 将输入文本转换为单词id列表
    token_ids = pad_or_truncate(token_ids, max_len) # 对单词id列表进行填充或截断，使其长度达到max_len

    x = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device) # 将单词id列表转换为PyTorch张量，并添加一个批次维度，移动到指定设备
    logits = model(x) # 前向传播，得到模型的输出logits
    probs = torch.softmax(logits, dim=1) # 对logits进行softmax

    pred_label = torch.argmax(probs, dim=1).item() # 获取概率最高的类别索引作为预测结果，并转换为Python整数
    return "Positive" if pred_label == 1 else "Negative" # 根据预测的标签返回对应的情感类别

##### 15.2 测试模型

In [33]:
sample_1 = "This movie was fantastic! I really enjoyed it."
sample_2 = "I hated this movie. It was terrible and a waste of time."

print(predict_sentiment(sample_1, model, max_len, device)) # 预测样本1的情感标签
print(predict_sentiment(sample_2, model, max_len, device)) # 预测样本2的情感标签

Positive
Negative
